# GBM long-horizon probe — do the EXISTING features carry direction at 15 min / 1 h?

**This is a probe, not a gated run.** It exists to answer one question cheaply,
before any two-stage notebook is built:

> The 90 s programme (runs 007-013) failed because the fee is larger than the move.
> On the stage-1 trigger set the required directional accuracy is **0.77 at 90 s**
> but only **0.61 at 15 min** and **0.59 at 1 h** (taker), because `E|move|` on
> triggers grows 18.6 -> 44.5 -> 55.8 bp while the fee stays at 10 bp.
> The volatility detector still concentrates magnitude 3.6x / 2.3x at those horizons.
> **So: is there any directional signal at 15 min - 1 h in the 76 existing features?**

Measured context (`runs/horizon_economics_and_next_ideas.md`, `run009f_scores.npz`):

| | 90 s | 5 min | 15 min | 1 h |
|---|---|---|---|---|
| IC of the *existing* LSTM `pred` | +0.046 | +0.034 | +0.025 | +0.006 |
| daily-IC t | +4.1 | +1.6 | +0.2 | -0.9 |
| stage-1 \|move\| lift | 4.63x | 4.33x | 3.59x | 2.27x |
| E\|move\| on triggers | 18.6 bp | 31.7 bp | 44.5 bp | 55.8 bp |
| **required accuracy @ taker 10 bp** | 0.769 | 0.658 | **0.612** | **0.590** |
| **required accuracy @ maker 4 bp** | 0.607 | 0.563 | **0.545** | **0.536** |

The existing LSTM's direction does **not** transfer past ~5 min, so a long-horizon
model has to be trained fresh. This notebook does that with gradient boosting
(fast to iterate; the LSTM overfits at epoch 1 in 10/12 seed-folds anyway).

## What it does

1. Inherits run.009f's **exact** feature pipeline — cells 1-5 are byte-identical,
   so the 76 features are the same objects, built the same way.
2. Builds **long-horizon targets** (15 min, 1 h) with a causal, horizon-scaled
   volatility normalisation.
3. Walk-forward over the **same 4 expanding folds**, with the purge widened from
   `MAX_H`(24) to `max(LONG_H)` so no long label reaches across a fold boundary.
4. Trains three GBMs per fold per horizon:
   - **stage 1 / `mag`** — predicts `|y|` (the big-move detector)
   - **stage 2a / `dir_all`** — predicts signed `y` on all bars
   - **stage 2b / `dir_evt`** — predicts signed `y` trained **only on eventful bars**
     (the "train direction on big moves only" hypothesis)
5. Reports IC, daily-IC t, directional accuracy **on the stage-1 trigger set**,
   net bp at 4 / 7 / 10 bp with day-clustered CIs, and gain importance with the
   13 `DROP_HARMFUL` + 5 `DROP_DEAD` features **put back in** (`PRUNE_MODE='none'`)
   so the prune can be re-judged on a direction objective instead of h18 AP.

## Pre-registered read-out

- **A (signal)** — `dir` daily-IC t >= `GATE_T` (3.0) at 15 min or 1 h, IC > 0 in >= 3/4 folds.
- **B (economics)** — directional accuracy on the stage-1 top-0.1% >= the required
  accuracy for the **maker** route at that horizon (0.545 @15 min, 0.536 @1 h).
- **C (net)** — net bp after 7 bp (realistic maker-in/taker-out) > 0 with a
  day-clustered CI excluding 0.
- **Falsification** — A fails at both horizons -> the existing bar features carry no
  usable long-horizon direction; do **not** build the two-stage notebook. Move to
  new inputs (run.013 price panel) or close the direction line.

A **pass on A but not B** is the informative middle: signal exists but is too weak
to trade, which is the run.009d-f pattern and must not be read as success.

## Runtime

GPU (T4) ~10-20 min, CPU ~40-80 min. Set `TRAIN_CAP` lower to trade accuracy for speed.
Needs `60days_data.tar` on Drive, same as run.009f.


In [ ]:
# ── Cell 1: Install and imports ───────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                'xgboost', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'scipy'],
               check=True)

import os, glob, gc, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
# ── Cell 2: (optional) data transfer helpers — uncomment what you need ────
#!apt install sshpass -y && cd ~ && sshpass -p 'XXXXXXXXXX' sftp -r -o "StrictHostKeyChecking no" scpuser@155.207.120.2:60days_data.tar .
#from google.colab import drive
#drive.mount('/content/drive')
#!mv btc_data.tar.xz /content/drive/MyDrive/
#!ls /content/drive/MyDrive/ -l
#drive.flush_and_unmount()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cd ~ && mkdir -p btc_data && cd btc_data && tar xvf /content/drive/MyDrive/60days_data.tar
#!cd ~/btc_data && tar xzvf ~/20day_btc_data.tar.gz
#drive.flush_and_unmount()


In [ ]:
# ── Cell 3: Config ─────────────────────────────────────────────────────────

# ── CONFIGURE THESE ───────────────────────────────────────────────────────
DATA_DIR   = '/root/btc_data'
OUTPUT_DIR = '/root/btc_gbm_probe'

WINDOW_SEC  = 5           # run.009a: the collector's w5 stream (v4.1 dual-
                          # writes w15+w5; the cell-5 loader filters on this)
HORIZONS    = [6, 9, 12, 15, 18, 24]
                          # run.009a: 5s bars — run.009's [2,3,4,5,6,8] ×3, i.e.
                          # the SAME wall-clock horizons 30s–2min. h24 (2min) is
                          # the anchor (same θ → lup_24/ldn_24 comparable to
                          # run.009's lup_8/ldn_8).
H_TRADE     = 24          # importance sampling / Huber weighting / reference IC
                          # follow the least-noisy kept horizon (run.004's
                          # t=+4.8 edge was at h8@15s = 2min = h24@5s)
VOL_WINDOW  = 720         # 1h rolling window for target normalisation (5s bars)
TARGET_CLIP = 5.0         # clip |target| at 5 sigma

SEQ_LEN      = 192        # 16 min of context (unchanged wall-clock; 3× bars at 5s)
HIDDEN_DIM   = 128        # SMALL on purpose — capacity was cleared twice (run.002/003)
NUM_LAYERS   = 2
DROPOUT      = 0.3
LR           = 1e-3
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 2048
EPOCHS       = 12
WARMUP       = 2          # linear warmup epochs before cosine decay
PATIENCE     = 5          # early stop on val AP of the two H_SEL opportunity heads
EPOCH_SAMPLE_CAP = 600_000  # train sequences drawn per epoch (weighted, w/ replacement)

# ── run.003 (kept): focus training on directional bars ──
SAMPLE_W_BASE = 0.25      # epoch-draw probability ∝ SAMPLE_W_BASE + |y_trade|
LOSS_W_ALPHA  = 1.0       # per-sample Huber weight = min(1 + α·|y_trade|, cap)
LOSS_W_CAP    = 4.0

# ── run.004 (kept): multi-seed evaluation ──
SEEDS = [0, 1, 2]         # per-fold; test score/prob = seed-ensemble mean
MIN_DAY_SAMPLES = 500     # a test day needs this many samples to enter daily-IC stats

N_FOLDS         = 4       # walk-forward folds over the last 60% of the window
TEST_START_FRAC = 0.40    # first 40% is never tested (training warmup)
VAL_FRAC        = 0.12    # tail of each training range used for early stopping
COVERAGE_MIN    = 0.60    # features with lower non-NaN coverage are excluded

MAKER_FEE = 0.0002        # per side (Binance USDT-M futures maker)
TAKER_FEE = 0.0005        # per side

# ── run.006 (kept): schema-3/4 era handling ──
V3_ERA_ONLY  = True      # restrict the analysis window to schema-3+ rows
V3_MARKER    = 'future_ofi_sum'   # first non-NaN value of this = era start
MIN_ERA_DAYS = 6         # abort if less schema-3+ data than this has accumulated
WALL_BAND    = 0.30       # collector WALL_BAND_PCT — wall dist −1 (=absent) maps here

# ── run.007: short-horizon big-move heads ──
H_SEL        = 18         # primary selective horizon (90s), pre-registered:
                          # 18×5s = 90s = run.009's h6 — wall-clock-unchanged.
                          # Rationale (run.007): closest kept horizon to run.006's
                          # best-lift h8, θ=20bp base rate stays trainable, and
                          # ~90% of a θ-touch survives to the 90s endpoint.
THETA_BY_H   = {6: 0.0015, 9: 0.0015,   # 15 bps at 30–45s: a 20 bps move inside
                12: 0.0020, 15: 0.0020, #   30–45s is near-unreachable (base ≲0.3%)
                18: 0.0020, 24: 0.0020} # 20 bps at 60–120s — same wall-clock map as run.009
POS_W_CAP    = 20.0       # BCE pos_weight = neg/pos per head, capped here
BCE_W        = 1.0        # classification loss weight vs the (frozen-form) Huber part

# ── run.007: TP/SL exit simulation (run.006's #1 lever, untested there) ──
TP_MULT      = 1.0        # primary take-profit = TP_MULT·θ_h (limit, maker exit)
SL_MULT      = 1.0        # primary stop = SL_MULT·θ_h (taker exit at the
                          # breaching bar's CLOSE — gap-through conservative)
TP_SL_GRID   = [(1.0, 1.0), (1.0, 0.5), (1.0, np.inf), (1.5, 1.0), (1.5, 0.5)]
                          # exploratory grid; ONLY (TP_MULT, SL_MULT) is gated

# ── run.007: rolling-causal trigger calibration ──
ROLL_CAL_DAYS = 14        # τ for test day D = (1−rate)-quantile of the scores
                          # seen in [D−14d, D): val seeds the window, already-
                          # seen test scores join it. Deployable (past data
                          # only); fixes run.006's fold-1 zero-trigger pathology.
MIN_CAL_N     = 5000      # thinner window → expand to all past scores

TRIGGER_RATES = [0.0001, 0.001, 0.01]  # nominal trigger rates
PRIMARY_RATE  = 0.001     # pre-registered primary: top-0.1% ≈ a few signals/day
N_BOOT        = 2000      # day-cluster bootstrap draws for net-bps CIs

# ── run.008: maker-entry simulation (run.007's diagnosed remaining lever) ──
ENTRY_WAIT   = 6          # bars the entry limit may rest before cancel
                          # (primary, pre-registered): 6×5s = 30s = run.009's
                          # wait=2 at 15s — wall-clock-unchanged; the pullback
                          # timing rationale (30s scale) is unchanged
ENTRY_DELTA  = 0.0        # limit offset in θ_h units below (up) / above (dn)
                          # the trigger close; 0 = post AT the signal close
ENTRY_DELAY  = 0          # bars between signal and posting (1 = latency stress)
WAIT_GRID    = [3, 6, 12, 18]      # exploratory (15/30/60/90s at 5s = run.009's grid)
DELTA_GRID   = [0.0, 0.25, 0.5]    # exploratory, ·θ_h
FILL_MIN     = 0.25       # gate C: primary config must fill ≥ this share of
                          # triggers — a maker edge that almost never fills is
                          # a statistical fluke, not a strategy

DRIVE_SAVE_DIR = '/content/drive/MyDrive/btc_gbm_probe'
# ─────────────────────────────────────────────────────────────────────────

# ── run.009d: feature pruning from the run.009b importance study ──────────
# run.009b permuted each of the 76 features against the fold-3 test set of the
# run.009a checkpoint (8 repeats, 20k windows) and measured the drop in h18
# up-head AP. 13 features had a SIGNIFICANTLY NEGATIVE drop - permuting them
# IMPROVED AP, the signature of harmful reliance - and 5 were constant zero in
# that window (no liquidations fired). run.009d drops them and retrains.
#
# The per-feature ranking is CORRELATION-BLIND, so this is a hypothesis to test,
# not a proven prune: run.009d's gate vs run.009a's numbers is the test.
# PRUNE_MODE='none' reproduces run.009a exactly (76 features).
PRUNE_MODE  = 'none'           # PROBE: keep all 76 - the prune was measured on
                               # h18 AP (a MAGNITUDE metric); this run re-judges it
                               # on a long-horizon DIRECTION metric.
DROP_BLOCKS = []               # block ablation: any of 'new_ctx', 'v3', 'v1_base'

DROP_HARMFUL = [
    'minute_sin', 'vol_norm', 'ma_gap_4h', 'dow_sin', 'basis_z_4h',
    'ma_gap_1h', 'sell_tail_ratio', 'wall_imbal', 'largest_trade_rel',
    'wall_qty_norm', 'flow_net_widex_z', 'buy_accel', 'minute_cos',
]
DROP_DEAD = [
    'liq_flag', 'liq_cnt_log', 'liq_imbal', 'liq_notional_log',
    'liq_notional_max_log',
]

# Exploratory training arm, OFF by default so the primary run stays
# single-variable. run.009a's validation IC peaked at epoch 1 in 10/12
# seed-folds and decayed with training - the model gets roughly one epoch of
# useful learning at LR 1e-3 - so 'regularized' slows it down instead.
TRAIN_ARM  = 'baseline'        # 'baseline' | 'regularized'
if TRAIN_ARM == 'regularized':
    LR = 2e-4
    DROPOUT = 0.4
EARLY_STOP = 'spread'           # 'ap' | 'ic' | 'dir' | 'spread' (run.009f: val MN spread)
DIR_RATE   = 0.02               # selection rate (2 % of val = 1,085–2,297 triggers/leg)
DIR_MIN    = 0.05               # gate B: pooled DE must reach this
MIN_TRIG_VAL = 500              # min triggers per leg for spread estimator

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Data dir:   {DATA_DIR}')
print(f'Output dir: {OUTPUT_DIR}')
print('θ per horizon: ' + '  '.join(f'h{h}={t*1e4:.0f}bp' for h, t in THETA_BY_H.items()))
print(f'taker RT = {2*TAKER_FEE*1e4:.0f} bps   TP RT (taker in, maker out) = '
      f'{(TAKER_FEE+MAKER_FEE)*1e4:.0f} bps   full-maker RT = '
      f'{2*MAKER_FEE*1e4:.0f} bps')


# ══════════════════════════════════════════════════════════════════════════
# GBM long-horizon probe — config
# ══════════════════════════════════════════════════════════════════════════
LONG_H     = [180, 720]        # 15 min and 1 h at 5 s bars
LAGS       = [12, 60, 180]     # feature deltas: 1 min / 5 min / 15 min
EVENT_RATE = 0.05              # 'eventful' share for the dir_evt training subset
TRIG_RATES = [0.001, 0.01]     # stage-1 trigger rates to evaluate
TRAIN_CAP  = 400_000           # max training rows per fold (RAM/speed); 0 = no cap
GATE_T     = 3.0               # criterion A: daily-IC t threshold
EARLY_ROUNDS = 50

GBM_PARAMS = dict(
    n_estimators=3000,         # capped by early stopping on the fold's val split
    max_depth=5,               # shallow on purpose: the signal is weak and noisy
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.5,
    min_child_weight=200,      # heavy leaf smoothing - financial targets are ~noise
    reg_lambda=2.0,
)

# Required directional accuracy on the stage-1 trigger set, p = (1 + F/E|move|)/2.
# E|move| measured from run009f_scores.npz (horizon_economics_and_next_ideas.md
# section 1 / section 6); recomputed from this run's own data in the economics cell.
EMOVE_TRIG_REF = {180: 44.46, 720: 55.82}   # bp, top-0.1% of the run.009f vol detector
FEE_ROUTES = [('maker', 4.0), ('mixed', 7.0), ('taker', 10.0)]

assert max(LAGS) <= SEQ_LEN - 1, 'a lag reaches past the SEQ_LEN contiguity guarantee'
print(f'\nPROBE: long horizons ' +
      '  '.join(f'h{h} ({h*WINDOW_SEC/60:.0f} min)' for h in LONG_H))
print(f'       lags {LAGS}  event rate {EVENT_RATE:.0%}  train cap {TRAIN_CAP:,}')


In [ ]:
# ── Cell 4: Feature pipeline ──────────────────────────────────────────────
SEPARATOR = ';'

COLS_NEEDED_BASE = {
    'spot_datetime', 'future_datetime', 'spot_timestamp', 'future_timestamp',
    'future_bid_close', 'future_ask_close',
    'future_bid_open', 'future_bid_max', 'future_bid_min',
    'future_bid_median', 'future_ask_median',
    'future_spread_open', 'future_spread_max',
    'future_buy_qty', 'future_sell_qty',
    'future_buy_samples', 'future_sell_samples',
    'future_buy_vwap', 'future_sell_vwap', 'future_price_diff',
    'future_bid_liq_0.0_median', 'future_ask_liq_0.0_median',
    'spot_buy_qty', 'spot_sell_qty',
    'spot_bid_close', 'spot_ask_close',
    'opt_open_interest_sample', 'opt_funding_rate_sample',
    'opt_est_funding_rate_sample', 'opt_remaining_time_sample',
    'opt_long_force_exit_qty_sum', 'opt_short_force_exit_qty_sum',
    'future_first_trade_side', 'future_last_trade_side',
    'future_largest_trade_qty', 'future_largest_trade_side',
    'future_buy_count_early', 'future_buy_count_late',
    'future_sell_count_early', 'future_sell_count_late',
    'future_buy_qty_early', 'future_buy_qty_late',
    'future_sell_qty_early', 'future_sell_qty_late',
}
COLS_NEEDED_LIQ_DEEP  = [
    'future_bid_liq_0.04_median', 'future_ask_liq_0.04_median',
    'future_bid_liq_0.05_median', 'future_ask_liq_0.05_median',
    'future_bid_liq_0.06_median', 'future_ask_liq_0.06_median',
]
COLS_NEEDED_LIQ_TOTAL = [
    'future_bid_liq_0.1_median',  'future_ask_liq_0.1_median',
    'future_bid_liq_0.2_median',  'future_ask_liq_0.2_median',
    'future_bid_liq_0.3_median',  'future_ask_liq_0.3_median',
    'future_bid_liq_0.4_median',  'future_ask_liq_0.4_median',
]
# run.006: schema-3/4 collector columns (out3/out4 files; NaN in older files)
COLS_NEEDED_V3 = [
    'future_mid_std', 'future_mid_rv', 'future_mid_flips',
    'future_micro_dev_close', 'future_micro_dev_median',
    'future_ofi_sum',
    'future_add_bid_near_sum', 'future_cancel_bid_near_sum',
    'future_add_ask_near_sum', 'future_cancel_ask_near_sum',
    'future_add_bid_wide_sum', 'future_cancel_bid_wide_sum',
    'future_add_ask_wide_sum', 'future_cancel_ask_wide_sum',
    'future_depth_msg_count', 'future_best_bid_move_count',
    'future_best_ask_move_count',
    'future_bid_depletion_count', 'future_ask_depletion_count',
    'future_buy_size_median', 'future_buy_size_p90',
    'future_sell_size_median', 'future_sell_size_p90',
    'future_bid_wall_qty_median', 'future_bid_wall_qty_max',
    'future_bid_wall_dist_median',
    'future_ask_wall_qty_median', 'future_ask_wall_qty_max',
    'future_ask_wall_dist_median',
    'future_max_batch_buy_qty', 'future_max_batch_sell_qty',
    'future_max_buy_run_qty', 'future_max_sell_run_qty',
    'opt_long_force_exit_cnt_sum', 'opt_short_force_exit_cnt_sum',
    'opt_long_force_exit_notional_sum', 'opt_short_force_exit_notional_sum',
    'opt_force_exit_notional_max',
    'opt_long_short_ratio_sample',
    'opt_eth_mid_open', 'opt_eth_mid_close',
    'schema_version',
]

# run.004: long-timescale context the 64-bar input window cannot see
NEW_FEATURES = [
    'ret_norm_1h', 'ret_norm_4h', 'ret_norm_24h',
    'ma_gap_1h', 'ma_gap_4h', 'ma_gap_24h',
    'vol_ratio_1h_24h', 'range_pos_24h',
    'basis_z_4h', 'basis_mom_1h',
    'dow_sin', 'dow_cos',
]

# run.006: engineered from the schema-3/4 columns. Episodic by nature —
# mostly quiet, occasionally scream — which is the shape a rare-trigger
# model needs (v1's continuous rolling stats are why runs 001–005 came
# out diffuse).
V3_FEATURES = [
    'ofi_z', 'flow_net_near_z', 'cancel_imbal_near',
    'pull_add_bid', 'pull_add_ask', 'flow_net_widex_z',
    'micro_dev_z', 'mid_rv_norm', 'mid_flips_norm', 'msg_rate_norm',
    'depl_imbal', 'depl_total_norm',
    'buy_tail_ratio', 'sell_tail_ratio',
    'liq_cnt_log', 'liq_imbal', 'liq_notional_log', 'liq_notional_max_log',
    'wall_imbal', 'bid_wall_dist', 'ask_wall_dist', 'wall_qty_norm',
    'batch_buy_rel', 'batch_sell_rel', 'sweep_buy_rel', 'sweep_sell_rel',
    'eth_ret_z', 'eth_lead_gap', 'lsr_z',
]

ALL_FEATURES = [
    'book_imbalance', 'flow_imbalance', 'momentum', 'book_imbal_deep',
    'flow_imbal_roll4', 'flow_imbal_roll8', 'book_imbal_roll4', 'book_imbal_roll8',
    'vwap_spread', 'liq_flag', 'stochastic', 'spread_expansion', 'sample_imbalance',
    'flow_agreement', 'oi_change', 'size_imbalance', 'liq_conc_bid', 'liq_conc_ask',
    'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos',
    'near_funding', 'funding_pressure', 'vol_norm',
    'trade_side_open', 'trade_side_close', 'trade_side_momentum',
    'largest_trade_side', 'largest_trade_rel',
    'buy_accel', 'sell_accel', 'flow_accel', 'buy_count_accel', 'late_imbalance',
] + NEW_FEATURES + V3_FEATURES

# binary opportunity heads: column order is frozen here and reused everywhere
CLS_KEYS = [(h, s) for h in HORIZONS for s in ('up', 'dn')]
CLS_COLS = [f'l{s}_{h}' for h, s in CLS_KEYS]
SEL_UP_J = CLS_KEYS.index((H_SEL, 'up'))
SEL_DN_J = CLS_KEYS.index((H_SEL, 'dn'))


def load_files(files):
    all_needed = (COLS_NEEDED_BASE | set(COLS_NEEDED_LIQ_DEEP)
                  | set(COLS_NEEDED_LIQ_TOTAL) | set(COLS_NEEDED_V3))
    frames = []
    for path in files:
        try:
            hdr = pd.read_csv(path, sep=SEPARATOR, nrows=0)
            usecols = sorted(all_needed & set(hdr.columns))
            if 'spot_datetime' not in usecols:
                print(f'  SKIP {os.path.basename(path)}: missing spot_datetime')
                continue
            df = pd.read_csv(path, sep=SEPARATOR, usecols=usecols, low_memory=False)
            df = df[df['spot_datetime'] != 'spot_datetime']
            frames.append(df)
        except Exception as e:
            print(f'  SKIP {os.path.basename(path)}: {e}')
    if not frames:
        raise RuntimeError('No valid CSV files found.')
    print(f'  Loaded {len(frames)} files')
    return pd.concat(frames, ignore_index=True)


def prepare(df, horizons, vol_window):
    df['dt'] = pd.to_datetime(df['spot_datetime'], errors='coerce')
    df = df.dropna(subset=['dt']).sort_values('dt').reset_index(drop=True)
    n_before = len(df)
    df = df.drop_duplicates(subset='dt', keep='first').reset_index(drop=True)
    if len(df) < n_before:
        print(f'  Dropped {n_before - len(df)} duplicate timestamps')

    skip = {'spot_datetime', 'future_datetime', 'dt'}
    for col in df.columns:
        if col not in skip:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    for col in [c for c in df.columns if c.startswith('opt_')]:
        df[col] = df[col].replace(-1, np.nan)

    if 'future_bid_close' in df.columns and 'future_ask_close' in df.columns:
        df['close_price'] = (df['future_bid_close'] + df['future_ask_close']) / 2.0
    else:
        df['close_price'] = (df['future_bid_median'] + df['future_ask_median']) / 2.0

    # Gap structure: collector restarts / dropped bars break the WINDOW_SEC cadence.
    gap = df['dt'].diff().dt.total_seconds()
    n_breaks = int((gap != WINDOW_SEC).sum()) - 1
    big = gap[gap > 3600]
    print(f'  Cadence breaks: {n_breaks}  (gaps >1h: {len(big)}, largest: {gap.max()/3600:.1f}h)')

    eps = 1e-9
    df['flow_imbalance'] = ((df['future_buy_qty'] - df['future_sell_qty']) /
                            (df['future_buy_qty'] + df['future_sell_qty'] + eps))
    bid0 = df['future_bid_liq_0.0_median']
    ask0 = df['future_ask_liq_0.0_median']
    df['book_imbalance'] = (bid0 - ask0) / (bid0 + ask0 + eps)
    df['momentum'] = df['future_price_diff']

    deep_col = None
    for lvl in ['0.05', '0.04', '0.06']:
        b, a = f'future_bid_liq_{lvl}_median', f'future_ask_liq_{lvl}_median'
        if b in df.columns and a in df.columns:
            deep_col = (b, a); break
    df['book_imbal_deep'] = ((df[deep_col[0]] - df[deep_col[1]]) /
                             (df[deep_col[0]] + df[deep_col[1]] + eps)) if deep_col else np.nan

    # run.009a (5s): bar-count rollers ×3 to keep run.009's wall-clock
    # meaning (60s/120s); feature NAMES kept so ALL_FEATURES is untouched
    df['flow_imbal_roll4'] = df['flow_imbalance'].rolling(12, min_periods=6).mean()
    df['flow_imbal_roll8'] = df['flow_imbalance'].rolling(24, min_periods=12).mean()
    df['book_imbal_roll4'] = df['book_imbalance'].rolling(12, min_periods=6).mean()
    df['book_imbal_roll8'] = df['book_imbalance'].rolling(24, min_periods=12).mean()

    df['vwap_spread'] = (df['future_buy_vwap'] - df['future_sell_vwap']
                         if 'future_buy_vwap' in df.columns else np.nan)
    liq_long  = df.get('opt_long_force_exit_qty_sum',  pd.Series(0, index=df.index))
    liq_short = df.get('opt_short_force_exit_qty_sum', pd.Series(0, index=df.index))
    df['liq_flag'] = ((liq_long + liq_short) > 0).astype(float)

    if all(c in df.columns for c in ['future_bid_max', 'future_bid_min', 'future_bid_close']):
        rng = df['future_bid_max'] - df['future_bid_min']
        df['stochastic'] = np.where(rng > 0, (df['future_bid_close'] - df['future_bid_min']) / rng, 0.5)
    else:
        df['stochastic'] = np.nan

    df['spread_expansion'] = (df['future_spread_max'] - df['future_spread_open']
                              if all(c in df.columns for c in ['future_spread_max', 'future_spread_open'])
                              else np.nan)

    if all(c in df.columns for c in ['future_buy_samples', 'future_sell_samples']):
        bs, ss = df['future_buy_samples'], df['future_sell_samples']
        df['sample_imbalance'] = (bs - ss) / (bs + ss + eps)
    else:
        df['sample_imbalance'] = np.nan

    if all(c in df.columns for c in ['spot_buy_qty', 'spot_sell_qty']):
        spot_flow = ((df['spot_buy_qty'] - df['spot_sell_qty']) /
                     (df['spot_buy_qty'] + df['spot_sell_qty'] + eps))
        df['flow_agreement'] = df['flow_imbalance'] * spot_flow
    else:
        df['flow_agreement'] = np.nan

    df['oi_change'] = df['opt_open_interest_sample'].diff() if 'opt_open_interest_sample' in df.columns else np.nan

    if all(c in df.columns for c in ['future_buy_samples', 'future_sell_samples',
                                      'future_buy_qty', 'future_sell_qty']):
        df['size_imbalance'] = (df['future_buy_qty']  / (df['future_buy_samples']  + eps) -
                                df['future_sell_qty'] / (df['future_sell_samples'] + eps))
    else:
        df['size_imbalance'] = np.nan

    sub_cols = ['future_first_trade_side', 'future_last_trade_side',
                'future_largest_trade_qty', 'future_largest_trade_side',
                'future_buy_count_early', 'future_buy_count_late',
                'future_sell_count_early', 'future_sell_count_late',
                'future_buy_qty_early', 'future_buy_qty_late',
                'future_sell_qty_early', 'future_sell_qty_late']
    if all(c in df.columns for c in sub_cols):
        bqe, bql = df['future_buy_qty_early'],  df['future_buy_qty_late']
        sqe, sql = df['future_sell_qty_early'],  df['future_sell_qty_late']
        fts, lts = df['future_first_trade_side'], df['future_last_trade_side']
        lq,  ls  = df['future_largest_trade_qty'], df['future_largest_trade_side']
        bce, bcl = df['future_buy_count_early'], df['future_buy_count_late']
        total_vol = bqe + bql + sqe + sql
        df['trade_side_open']     = fts
        df['trade_side_close']    = lts
        df['trade_side_momentum'] = lts - fts
        df['largest_trade_side']  = ls
        df['largest_trade_rel']   = lq / (total_vol + eps)
        df['buy_accel']           = bql - bqe
        df['sell_accel']          = sql - sqe
        df['flow_accel']          = (bql - sql) - (bqe - sqe)
        df['buy_count_accel']     = bcl - bce
        df['late_imbalance']      = (bql - sql) / (bql + sql + eps)
    else:
        for c in ['trade_side_open','trade_side_close','trade_side_momentum',
                  'largest_trade_side','largest_trade_rel','buy_accel','sell_accel',
                  'flow_accel','buy_count_accel','late_imbalance']:
            df[c] = np.nan

    deep_bid = next((c for c in ['future_bid_liq_0.4_median','future_bid_liq_0.3_median',
                                  'future_bid_liq_0.2_median','future_bid_liq_0.1_median']
                     if c in df.columns), None)
    deep_ask = next((c for c in ['future_ask_liq_0.4_median','future_ask_liq_0.3_median',
                                  'future_ask_liq_0.2_median','future_ask_liq_0.1_median']
                     if c in df.columns), None)
    df['liq_conc_bid'] = (df['future_bid_liq_0.0_median'] / (df[deep_bid] + eps)
                          if deep_bid else np.nan)
    df['liq_conc_ask'] = (df['future_ask_liq_0.0_median'] / (df[deep_ask] + eps)
                          if deep_ask else np.nan)

    hour, minute = df['dt'].dt.hour, df['dt'].dt.minute
    df['hour_sin']   = np.sin(2 * np.pi * hour   / 24)
    df['hour_cos']   = np.cos(2 * np.pi * hour   / 24)
    df['minute_sin'] = np.sin(2 * np.pi * minute / 60)
    df['minute_cos'] = np.cos(2 * np.pi * minute / 60)

    secs = (hour * 3600 + minute * 60 + df['dt'].dt.second) % (8 * 3600)
    df['near_funding'] = ((8 * 3600 - secs) < 900).astype(float)
    if 'opt_funding_rate_sample' in df.columns:
        funding = df['opt_funding_rate_sample'].replace(-1, np.nan).fillna(0)
        df['funding_pressure'] = funding * df['near_funding']
    else:
        df['funding_pressure'] = np.nan

    # run.009a (5s): 4-min volatility (48 bars) vs its 24h mean (17280 bars)
    df['volatility'] = df['close_price'].diff().abs().rolling(48, min_periods=12).mean()
    df['vol_norm']   = (df['volatility'] /
                        df['volatility'].rolling(17280, min_periods=288).mean()
                       ).replace([np.inf, -np.inf], np.nan)

    # ── run.004: long-timescale context features ─────────────────────────
    # All rolling windows are TIME-based on the datetime index (rolling('24h')),
    # not row-count based: collector restarts land on arbitrary second offsets
    # and gaps would otherwise stretch a "1h" window over more wall time.
    # Everything is causal (past data only) and vol-normalised so the scale is
    # stationary across regimes; clipped at ±8 to bound scaler-era outliers.
    D = 24 * 3600 // WINDOW_SEC              # bars per 24h at full cadence
    lp = pd.Series(np.log(df['close_price'].values), index=df['dt'])

    def past(series, delta, tol_sec=2 * WINDOW_SEC):
        # value of `series` ~delta earlier (nearest bar within tol), aligned back
        p = series.reindex(series.index - delta, method='nearest',
                           tolerance=pd.Timedelta(seconds=tol_sec))
        return pd.Series(p.values, index=series.index)

    r1 = lp.diff()
    r1[gap.values != WINDOW_SEC] = np.nan    # 1-bar return across a gap is not 1-bar
    sigma24 = r1.rolling('24h', min_periods=D // 4).std()

    for name, td, W in [('1h', pd.Timedelta(hours=1),  D // 24),
                        ('4h', pd.Timedelta(hours=4),  D // 6),
                        ('24h', pd.Timedelta(hours=24), D)]:
        unit = sigma24 * np.sqrt(W) + eps
        df[f'ret_norm_{name}'] = ((lp - past(lp, td)) / unit).clip(-8, 8).values
        ma = lp.rolling(td, min_periods=int(W * 0.75)).mean()
        df[f'ma_gap_{name}'] = ((lp - ma) / unit).clip(-8, 8).values

    vol1h = r1.rolling('1h', min_periods=D // 48).std()
    df['vol_ratio_1h_24h'] = (vol1h / (sigma24 + eps)).clip(0, 5).values

    hi24 = lp.rolling('24h', min_periods=D // 4).max()
    lo24 = lp.rolling('24h', min_periods=D // 4).min()
    df['range_pos_24h'] = ((lp - lo24) / (hi24 - lo24 + eps)).clip(0, 1).values

    if all(c in df.columns for c in ['spot_bid_close', 'spot_ask_close']):
        spot_mid = (df['spot_bid_close'] + df['spot_ask_close']) / 2.0
        basis = pd.Series(((df['close_price'] - spot_mid) / (spot_mid + eps)).values,
                          index=df['dt'])
        b_ma = basis.rolling('4h', min_periods=D // 12).mean()
        b_sd = basis.rolling('4h', min_periods=D // 12).std()
        df['basis_z_4h'] = ((basis - b_ma) / (b_sd + eps)).clip(-8, 8).values
        b_mom = basis - past(basis, pd.Timedelta(hours=1))
        b_mom_sd = b_mom.rolling('4h', min_periods=D // 12).std()
        df['basis_mom_1h'] = (b_mom / (b_mom_sd + eps)).clip(-8, 8).values
    else:
        df['basis_z_4h'] = np.nan
        df['basis_mom_1h'] = np.nan

    dow = df['dt'].dt.dayofweek
    df['dow_sin'] = np.sin(2 * np.pi * dow / 7)
    df['dow_cos'] = np.cos(2 * np.pi * dow / 7)

    # ── run.006: schema-3/4 microstructure features ──────────────────────
    # Same conventions as the run.004 block: causal time-based rollers,
    # z-scores clipped ±8, ratios clipped [0, 10]. Every group is guarded on
    # column presence so pre-v3 files simply yield NaN (the era slice and the
    # coverage guard in the next cell handle the rest).
    mp1h = D // 24                                    # 1h of bars = min_periods

    def _ts(x):
        return pd.Series(np.asarray(x, dtype=np.float64), index=df['dt'])

    def roll_z(x, win='4h', mp=mp1h):
        s = _ts(x)
        return ((s - s.rolling(win, min_periods=mp).mean()) /
                (s.rolling(win, min_periods=mp).std() + eps)).clip(-8, 8).values

    def roll_ratio(x, win='4h', mp=mp1h, hi=10.0):
        s = _ts(x)
        return (s / (s.rolling(win, min_periods=mp).mean() + eps)).clip(0, hi).values

    have = set(df.columns)

    df['ofi_z'] = roll_z(df['future_ofi_sum']) if 'future_ofi_sum' in have else np.nan

    flow_near = ['future_add_bid_near_sum', 'future_cancel_bid_near_sum',
                 'future_add_ask_near_sum', 'future_cancel_ask_near_sum']
    if set(flow_near) <= have:
        ab, cb = df[flow_near[0]], df[flow_near[1]]
        aa, ca = df[flow_near[2]], df[flow_near[3]]
        df['flow_net_near_z']   = roll_z((ab - cb) - (aa - ca))
        df['cancel_imbal_near'] = ((cb - ca) / (cb + ca + eps)).values
        # bid pulled faster than replenished = support evaporating (log-diff
        # is scale-free, so no rolling normalisation needed)
        df['pull_add_bid'] = (np.log1p(cb) - np.log1p(ab)).clip(-8, 8)
        df['pull_add_ask'] = (np.log1p(ca) - np.log1p(aa)).clip(-8, 8)
    else:
        for c in ['flow_net_near_z', 'cancel_imbal_near', 'pull_add_bid', 'pull_add_ask']:
            df[c] = np.nan

    flow_wide = ['future_add_bid_wide_sum', 'future_cancel_bid_wide_sum',
                 'future_add_ask_wide_sum', 'future_cancel_ask_wide_sum']
    if set(flow_wide) <= have and set(flow_near) <= have:
        # the wide band INCLUDES the near band by construction — subtract to
        # get the 0.05–0.25% ring only (schema-4 files; NaN in out3 hours)
        abx = (df[flow_wide[0]] - df[flow_near[0]]).clip(lower=0)
        cbx = (df[flow_wide[1]] - df[flow_near[1]]).clip(lower=0)
        aax = (df[flow_wide[2]] - df[flow_near[2]]).clip(lower=0)
        cax = (df[flow_wide[3]] - df[flow_near[3]]).clip(lower=0)
        df['flow_net_widex_z'] = roll_z((abx - cbx) - (aax - cax))
    else:
        df['flow_net_widex_z'] = np.nan

    df['micro_dev_z'] = (roll_z(df['future_micro_dev_close'])
                         if 'future_micro_dev_close' in have else np.nan)
    df['mid_rv_norm']    = roll_ratio(df['future_mid_rv'])    if 'future_mid_rv'    in have else np.nan
    df['mid_flips_norm'] = roll_ratio(df['future_mid_flips']) if 'future_mid_flips' in have else np.nan
    df['msg_rate_norm']  = (roll_ratio(df['future_depth_msg_count'])
                            if 'future_depth_msg_count' in have else np.nan)

    if {'future_bid_depletion_count', 'future_ask_depletion_count'} <= have:
        bd, ad = df['future_bid_depletion_count'], df['future_ask_depletion_count']
        df['depl_imbal']      = ((bd - ad) / (bd + ad + 1.0)).values
        df['depl_total_norm'] = roll_ratio(bd + ad)
    else:
        df['depl_imbal'] = np.nan
        df['depl_total_norm'] = np.nan

    for side in ('buy', 'sell'):
        med, p90 = f'future_{side}_size_median', f'future_{side}_size_p90'
        if {med, p90} <= have:
            # fat right tail at equal volume = informed flow; log1p of the
            # ratio is scale-free across the BTC-qty drift
            df[f'{side}_tail_ratio'] = np.log1p((df[p90] / (df[med] + eps)).clip(0, 100))
        else:
            df[f'{side}_tail_ratio'] = np.nan

    # liquidations: an absent liquidation IS zero, so cnt/notional NaNs are
    # filled with 0 — but only inside the schema-3+ era, so the coverage
    # guard still sees pre-v3 rows as missing.
    m3 = df['future_ofi_sum'].notna() if 'future_ofi_sum' in have else pd.Series(False, index=df.index)
    liq_cols = ['opt_long_force_exit_cnt_sum', 'opt_short_force_exit_cnt_sum',
                'opt_long_force_exit_notional_sum', 'opt_short_force_exit_notional_sum',
                'opt_force_exit_notional_max']
    for c in liq_cols:
        if c in have:
            df.loc[m3, c] = df.loc[m3, c].fillna(0)
    if {'opt_long_force_exit_cnt_sum', 'opt_short_force_exit_cnt_sum'} <= have:
        df['liq_cnt_log'] = np.log1p(df['opt_long_force_exit_cnt_sum'] +
                                     df['opt_short_force_exit_cnt_sum'])
    else:
        df['liq_cnt_log'] = np.nan
    lqf = df.get('opt_long_force_exit_qty_sum',  pd.Series(np.nan, index=df.index)).fillna(0)
    sqf = df.get('opt_short_force_exit_qty_sum', pd.Series(np.nan, index=df.index)).fillna(0)
    df['liq_imbal'] = ((lqf - sqf) / (lqf + sqf + eps)).values
    if {'opt_long_force_exit_notional_sum', 'opt_short_force_exit_notional_sum'} <= have:
        df['liq_notional_log'] = np.log1p(df['opt_long_force_exit_notional_sum'] +
                                          df['opt_short_force_exit_notional_sum'])
    else:
        df['liq_notional_log'] = np.nan
    df['liq_notional_max_log'] = (np.log1p(df['opt_force_exit_notional_max'])
                                  if 'opt_force_exit_notional_max' in have else np.nan)

    wall_cols = ['future_bid_wall_qty_median', 'future_ask_wall_qty_median',
                 'future_bid_wall_dist_median', 'future_ask_wall_dist_median']
    if set(wall_cols) <= have:
        bq, aq = df[wall_cols[0]], df[wall_cols[1]]
        df['wall_imbal'] = ((bq - aq) / (bq + aq + eps)).values
        # dist −1 = no level in band → map to the band edge ("wall absent" =
        # "as far away as possible"); qty 0 already carries the absence signal
        df['bid_wall_dist'] = df[wall_cols[2]].where(df[wall_cols[2]] >= 0, WALL_BAND)
        df['ask_wall_dist'] = df[wall_cols[3]].where(df[wall_cols[3]] >= 0, WALL_BAND)
        df['wall_qty_norm'] = roll_ratio(bq + aq)
    else:
        for c in ['wall_imbal', 'bid_wall_dist', 'ask_wall_dist', 'wall_qty_norm']:
            df[c] = np.nan

    if {'future_max_batch_buy_qty', 'future_max_buy_run_qty'} <= have:
        # concentration of the bar's volume in one 200ms burst / one same-side
        # sweep run — bounded [0,1] per side, no normalisation needed
        df['batch_buy_rel']  = (df['future_max_batch_buy_qty']  / (df['future_buy_qty']  + eps)).clip(0, 1)
        df['batch_sell_rel'] = (df['future_max_batch_sell_qty'] / (df['future_sell_qty'] + eps)).clip(0, 1)
        df['sweep_buy_rel']  = (df['future_max_buy_run_qty']    / (df['future_buy_qty']  + eps)).clip(0, 1)
        df['sweep_sell_rel'] = (df['future_max_sell_run_qty']   / (df['future_sell_qty'] + eps)).clip(0, 1)
    else:
        for c in ['batch_buy_rel', 'batch_sell_rel', 'sweep_buy_rel', 'sweep_sell_rel']:
            df[c] = np.nan

    if {'opt_eth_mid_open', 'opt_eth_mid_close'} <= have:
        eo, ec = df['opt_eth_mid_open'], df['opt_eth_mid_close']
        eth_ret = pd.Series(np.where((eo > 0) & (ec > 0), np.log(ec / eo), np.nan),
                            index=df['dt'])
        df['eth_ret_z'] = (eth_ret / (eth_ret.rolling('4h', min_periods=mp1h).std() + eps)
                           ).clip(-8, 8).values
        # ETH ran but BTC has not (yet): 1-min cumulative z-gap
        # (run.009a: 12 bars = 1 min at 5s)
        er4 = eth_ret.rolling(12, min_periods=12).sum()
        br4 = r1.rolling(12, min_periods=12).sum()
        er4z = er4 / (er4.rolling('4h', min_periods=mp1h).std() + eps)
        br4z = br4 / (br4.rolling('4h', min_periods=mp1h).std() + eps)
        df['eth_lead_gap'] = (er4z - br4z).clip(-8, 8).values
    else:
        df['eth_ret_z'] = np.nan
        df['eth_lead_gap'] = np.nan

    df['lsr_z'] = (roll_z(df['opt_long_short_ratio_sample'], '24h', D // 4)
                   if 'opt_long_short_ratio_sample' in have else np.nan)

    # Scattered-NaN guard: a sample needs SEQ_LEN contiguous surviving rows behind
    # it and MAX_H ahead, so a feature with even ~2% RANDOMLY scattered NaNs
    # (an ETH bar with no data, a missing long/short-ratio sample, a 4-bar
    # rolling gap echo) would wipe out most valid samples via dropna. These
    # per-bar signals are neutral-at-0 by construction (z-scores / gaps), so
    # fill them with 0 inside the schema-3+ era; warmup and pre-era rows stay
    # NaN so the coverage guard still sees them.
    for c in ['eth_ret_z', 'eth_lead_gap', 'lsr_z']:
        bad = m3 & df[c].isna() & (df['dt'] > df.loc[m3, 'dt'].iloc[0] +
                                   pd.Timedelta('24h')) if m3.any() else None
        if bad is not None:
            df.loc[bad, c] = 0.0

    del lp, r1, sigma24, vol1h, hi24, lo24
    # ──────────────────────────────────────────────────────────────────────

    # Targets: volatility-normalised future return per horizon. A target is
    # valid only when the row h bars ahead really is h*WINDOW_SEC ahead — no labels
    # across collector gaps. vol uses only past prices (no leakage).
    # (datetime64[s] cast is robust to the pandas 2/3 resolution change)
    dt_sec = pd.Series(df['dt'].values.astype('datetime64[s]').astype(np.int64),
                       index=df.index)
    for h in horizons:
        vol   = df['close_price'].diff(h).rolling(vol_window, min_periods=vol_window // 4).std()
        delta = df['close_price'].shift(-h) - df['close_price']
        y     = (delta / (vol + eps)).clip(-TARGET_CLIP, TARGET_CLIP)
        valid = (dt_sec.shift(-h) - dt_sec) == h * WINDOW_SEC
        y[~valid.fillna(False)] = np.nan
        df[f'y_{h}'] = y

    # ── run.009: triple-barrier FIRST-TOUCH labels ───────────────────────
    # run.006/007/008 used a pure MFE-touch label (lup=1 if +θ reached
    # ANYWHERE in the next h bars, IGNORING whether −θ was hit first) —
    # mislabelling exactly the paths the TP/SL sim loses on (a dip to −θ is
    # a taker SL fill at −30 bp; a later recovery to +θ was still counted a
    # "win"). run.008's gate failed via adverse selection (hit_m ≫ hit_f);
    # this aligns label↔payoff: lup_h = 1 iff +θ is touched STRICTLY BEFORE
    # −θ within the next h bars; ldn_h mirrors. Neither-touched or a tie →
    # 0/0 ("no idea"). Same h·WINDOW_SEC gap-validity rule as the regression
    # targets (an exact h·WINDOW_SEC span to row i+h implies every intermediate bar
    # is present). Vectorized (n×h booleans, h≤24 → RAM-safe); validated ==
    # per-row brute force over 3000 randomized gapped paths (test_first_touch.py).
    cvals = df['close_price'].values.astype(np.float64)
    n_rows = len(cvals)
    for h in horizons:
        lup = np.full(n_rows, np.nan)
        ldn = np.full(n_rows, np.nan)
        if n_rows > h:
            wv   = np.lib.stride_tricks.sliding_window_view(cvals, h)[1:]  # wv[i]=cvals[i+1:i+1+h]
            base = cvals[:n_rows - h][:, None]
            th   = THETA_BY_H[h]
            up_hit = (wv / base - 1.0) >= th          # (n_win × h) booleans
            dn_hit = (1.0 - wv / base) >= th
            t_up = np.where(up_hit.any(1), up_hit.argmax(1), h)   # first touch bar; h = never
            t_dn = np.where(dn_hit.any(1), dn_hit.argmax(1), h)
            lup[:n_rows - h] = (t_up < t_dn).astype(np.float32)   # up strictly first
            ldn[:n_rows - h] = (t_dn < t_up).astype(np.float32)   # dn strictly first
        valid = ((dt_sec.shift(-h) - dt_sec) == h * WINDOW_SEC).fillna(False).values
        df[f'lup_{h}'] = np.where(valid, lup, np.nan)
        df[f'ldn_{h}'] = np.where(valid, ldn, np.nan)
    return df

print('Feature pipeline loaded.')
print(f'Binary heads: {CLS_COLS}')


In [ ]:
# ── Cell 5: Load data, era slice, coverage guard, build arrays ────────────
# run.009a: ONLY the w{WINDOW_SEC}s stream — the v4.1 collector dual-writes
# w15 and w5 files; without this filter a mixed tar would silently interleave
# the two cadences (sorting by dt mixes 5s and 15s bars into garbage).
files = sorted(f for f in (glob.glob(os.path.join(DATA_DIR, '*.csv')) +
                           glob.glob(os.path.join(DATA_DIR, '*.csv.gz')))
               if f'.w{WINDOW_SEC}.' in os.path.basename(f))
print(f'Found {len(files)} files (w{WINDOW_SEC} only)')

raw = load_files(files)
print(f'Total rows loaded: {len(raw):,}')

df = prepare(raw, HORIZONS, VOL_WINDOW)
del raw; gc.collect()

# ── run.006: restrict the analysis window to the schema-3+ era ───────────
# The v3 features only exist where the v3/v4 collector ran; over the full
# ~13-month range they would all fail the coverage guard. Features are
# computed on the FULL frame first (so the 24h rollers are warm at the era
# boundary), then the frame is sliced.
if V3_ERA_ONLY:
    mark = (df[V3_MARKER].notna() if V3_MARKER in df.columns
            else pd.Series(False, index=df.index))
    assert mark.any(), (
        f'No schema-3 data found ({V3_MARKER} entirely missing). Either the '
        f'out3/out4 files are not in btc_data.tar.xz — rebuild the tar — or '
        f'set V3_ERA_ONLY = False to run on v1 features only.')
    v3_start = df.loc[mark, 'dt'].iloc[0]
    era_days = (df['dt'].iloc[-1] - v3_start).total_seconds() / 86400
    n_pre = int((df['dt'] < v3_start).sum())
    print(f'\nSchema-3+ era starts {v3_start}  ({era_days:.1f} days, '
          f'dropping {n_pre:,} pre-era rows)')
    assert era_days >= MIN_ERA_DAYS, (
        f'Only {era_days:.1f} days of schema-3+ data (< {MIN_ERA_DAYS}). '
        f'Let the collector accumulate more before running run.006.')
    df = df[df['dt'] >= v3_start].reset_index(drop=True)
    if 'schema_version' in df.columns:
        sv = df['schema_version']
        print('  schema mix: ' + '  '.join(
            f'v{int(v)}: {(sv == v).mean()*100:.1f}%' for v in sorted(sv.dropna().unique())))

# Coverage guard (within the analysis window): schema-4-only columns (wide
# band, walls, bursts, ETH) are NaN in out3 hours; anything below
# COVERAGE_MIN is excluded rather than letting dropna() eat the window.
features = []
for f_ in ALL_FEATURES:
    if f_ not in df.columns:
        continue
    cov = df[f_].notna().mean()
    if cov >= COVERAGE_MIN:
        features.append(f_)
    else:
        print(f'  EXCLUDED {f_:22s} coverage {cov*100:5.1f}% < {COVERAGE_MIN*100:.0f}%')
# ── run.009e: apply the feature prune (from run.009d) ──────────────────────────────────────
_drop = set()
if PRUNE_MODE in ('harmful', 'harmful+dead'):
    _drop |= set(DROP_HARMFUL)
if PRUNE_MODE == 'harmful+dead':
    _drop |= set(DROP_DEAD)
if DROP_BLOCKS:
    _v1_base = set(features) - set(NEW_FEATURES) - set(V3_FEATURES)
    for _b in DROP_BLOCKS:
        _drop |= {'new_ctx': set(NEW_FEATURES), 'v3': set(V3_FEATURES),
                  'v1_base': _v1_base}.get(_b, set())
if _drop:
    _n_before = len(features)
    features = [f_ for f_ in features if f_ not in _drop]
    print(f'run.009f prune (from run.009d): mode={PRUNE_MODE} blocks={DROP_BLOCKS}  '
          f'{_n_before} -> {len(features)} features ({_n_before - len(features)} dropped)')
    assert len(features) >= 20, 'prune removed too many features - check the drop lists'
kept_v3 = [f_ for f_ in features if f_ in V3_FEATURES]
print(f'Features kept: {len(features)}/{len(ALL_FEATURES)} '
      f'(schema-3/4: {len(kept_v3)}/{len(V3_FEATURES)})')
# run.006 is built to test the schema-3/4 features, but it degrades to the
# v1 feature subset when those columns are absent (v1 collector files, or a
# btc_data.tar.xz that dropped the out3/out4 files) instead of aborting.
V3_AVAILABLE = len(kept_v3) >= 10
if not V3_AVAILABLE:
    print(f'\n*** v1-compat mode: only {len(kept_v3)}/{len(V3_FEATURES)} schema-3/4 '
          f'features present ***')
    print('    The loaded files predate the schema-3/4 collector (no future_ofi_sum, '
          'walls, ETH lead-lag, ...),')
    print('    so the selective/opportunity heads train on the v1 feature subset only. '
          'This does NOT')
    print('    test run.006\'s schema-3/4 hypothesis — it reruns the run.004/005 feature '
          'set under the')
    print('    selective objective. Feed out3/out4 files (rebuild btc_data.tar.xz) for '
          'the intended run.')

n0 = len(df)
clean = df[features + [f'y_{h}' for h in HORIZONS] + CLS_COLS + ['dt', 'close_price']] \
          .dropna(subset=features).reset_index(drop=True)
lost = n0 - len(clean)
print(f'Rows: {n0:,} → {len(clean):,} after dropna ({lost/max(n0,1)*100:.2f}% dropped)')
if lost > 0.05 * n0:
    print('\n*** WARNING: dropna removed >5% of rows — check per-feature NaN fractions: ***')
    print(df[features].isna().mean().sort_values(ascending=False).head(10))
del df; gc.collect()

X_raw  = clean[features].values.astype(np.float32)
Y_all  = np.stack([clean[f'y_{h}'].values for h in HORIZONS], axis=1).astype(np.float32)
L_all  = np.stack([clean[c].values for c in CLS_COLS], axis=1).astype(np.float32)
dt_s   = clean['dt'].values.astype('datetime64[s]').astype(np.int64)
dt_all = clean['dt'].values
close  = clean['close_price'].values.astype(np.float64)
n, F_DIM = X_raw.shape
del clean; gc.collect()

# A sample "ends" at row i: input window = rows [i-SEQ_LEN+1, i], targets and
# labels at/through i+h. Backward validity: the window must be contiguous
# WINDOW_SEC-spaced bars. Forward validity: row i+MAX_H exactly MAX_H·WINDOW_SEC ahead IN THIS
# (post-dropna) array — dropna can remove interior rows, and the evaluation
# below indexes close[e + h] positionally, so the check must hold here, not
# just in the pre-drop frame. (Improvement over run.005, which relied on the
# pre-drop check only.)
MAX_H = max(HORIZONS)
contig = np.zeros(n, dtype=bool)
span = (SEQ_LEN - 1) * WINDOW_SEC
contig[SEQ_LEN-1:] = (dt_s[SEQ_LEN-1:] - dt_s[:n-SEQ_LEN+1]) == span
fwd_ok = np.zeros(n, dtype=bool)
fwd_ok[:n-MAX_H] = (dt_s[MAX_H:] - dt_s[:n-MAX_H]) == MAX_H * WINDOW_SEC
sample_valid = (contig & fwd_ok &
                np.isfinite(Y_all).all(axis=1) & np.isfinite(L_all).all(axis=1))
assert sample_valid.sum() > 0, 'No valid samples — check bar cadence / targets'

print(f'\nRows: {n:,}   valid samples: {sample_valid.sum():,} ({sample_valid.mean()*100:.1f}%)')
print(f'Date range: {pd.Timestamp(dt_all[0])} → {pd.Timestamp(dt_all[-1])}')
H_TRADE_IDX = HORIZONS.index(H_TRADE)
yt = Y_all[sample_valid, H_TRADE_IDX]
print(f'Trading target y_{H_TRADE}: std={yt.std():.3f}   |y|>1σ: {(np.abs(yt) > 1).mean()*100:.1f}%   '
      f'up share: {(yt > 0).mean()*100:.1f}%')
print('\nOpportunity-label base rates (per-horizon θ, valid samples):')
for j, (h, s) in enumerate(CLS_KEYS):
    print(f'  {CLS_COLS[j]:8s} θ={THETA_BY_H[h]*1e4:2.0f}bp  '
          f'{L_all[sample_valid, j].mean()*100:6.2f}%')


In [ ]:
# ── Cell 6: Walk-forward folds (expanding window, purged) ─────────────────
# PROBE: the purge must cover the LONGEST label, not MAX_H(=24). A 1 h label
# started just before a fold boundary resolves 720 bars later, i.e. inside the
# next split — with the run.009f purge that is direct label leakage into test.
PURGE = max(MAX_H, max(LONG_H))
print(f'purge = {PURGE} bars ({PURGE*WINDOW_SEC/60:.0f} min)  '
      f'[run.009f used MAX_H = {MAX_H}]')

def ends_in(a, b):
    return np.flatnonzero(sample_valid[a:b]) + a

test_start = int(n * TEST_START_FRAC)
chunk = (n - test_start) // N_FOLDS
folds = []
for k in range(N_FOLDS):
    lo = test_start + k * chunk
    hi = n if k == N_FOLDS - 1 else lo + chunk
    boundary = lo - PURGE                     # purge: val labels must not reach into test
    val_lo   = int(boundary * (1 - VAL_FRAC))
    folds.append({'k': k,
                  'train_hi': val_lo - PURGE,  # purge: train labels must not reach into val
                  'val':  (val_lo, boundary),
                  'test': (lo, hi)})

print('Fold   n_train     n_val    n_test    test period')
for f_ in folds:
    t_lo, t_hi = f_['test']
    n_tr = len(ends_in(0, f_['train_hi']))
    n_va = len(ends_in(*f_['val']))
    n_te = len(ends_in(t_lo, t_hi))
    assert min(n_tr, n_va, n_te) > 0, f"fold {f_['k']} has an empty split"
    span_d = (dt_s[t_hi-1] - dt_s[t_lo]) / 86400
    warn = '   *** <7 days — treat this fold as indicative only ***' if span_d < 7 else ''
    print(f"  {f_['k']}  {n_tr:>9,}  {n_va:>8,}  {n_te:>8,}    "
          f"{pd.Timestamp(dt_all[t_lo])} → {pd.Timestamp(dt_all[t_hi-1])} "
          f"({span_d:.1f}d){warn}")


## Probe — long-horizon targets, GBM walk-forward, economics

In [ ]:
# ── Cell G1: long-horizon targets ─────────────────────────────────────────
# Built here rather than inside prepare() so cells 1-5 stay byte-identical to
# run.009f and MAX_H / sample_valid / contig keep exactly their run.009f meaning.
#
#   Y_LONG[h][e] = (close[e+h] - close[e]) / vol_h[e]   vol_h uses PAST bars only
#   RAW_BP[h][e] = (close[e+h]/close[e] - 1) * 1e4      for the economics cell
#
# vol_h mirrors prepare()'s own formula (rolling std of the h-bar price change)
# with a horizon-scaled window, because a 720-bar diff inside a 720-bar window
# would be a single overlapping observation.

cser = pd.Series(close)
Y_LONG, RAW_BP, LONG_OK = {}, {}, {}

print(f'{"h":>6} {"wall":>8} {"vol win":>9} {"valid":>12} {"E|move|":>9} {"sd(y)":>7} {"up%":>6}')
for h in LONG_H:
    vol_n = max(VOL_WINDOW, 8 * h)
    vol = cser.diff(h).rolling(vol_n, min_periods=vol_n // 4).std().values

    ok = np.zeros(n, dtype=bool)
    ok[:n - h] = (dt_s[h:] - dt_s[:n - h]) == h * WINDOW_SEC   # no label across a gap

    dpx = np.full(n, np.nan); dpx[:n - h] = close[h:] - close[:n - h]
    rbp = np.full(n, np.nan); rbp[:n - h] = (close[h:] / close[:n - h] - 1.0) * 1e4
    y = np.clip(dpx / (vol + 1e-12), -TARGET_CLIP, TARGET_CLIP)

    good = ok & np.isfinite(y) & np.isfinite(vol) & (vol > 0)
    y[~good] = np.nan; rbp[~good] = np.nan
    Y_LONG[h], RAW_BP[h], LONG_OK[h] = y, rbp, good

    print(f'{h:>6} {h*WINDOW_SEC/60:>7.0f}m {vol_n:>9,} {good.sum():>12,} '
          f'{np.nanmean(np.abs(rbp)):>8.2f}b {np.nanstd(y[good]):>7.3f} '
          f'{(rbp[good] > 0).mean()*100:>5.1f}%')

# Sanity: a long target must never be finite where the forward span is broken.
for h in LONG_H:
    assert not np.isfinite(Y_LONG[h][~LONG_OK[h]]).any(), f'h{h}: target survives a gap'
print('\ngap-validity assertion passed for all long horizons')


In [ ]:
# ── Cell G2: GBM design matrix ────────────────────────────────────────────
# A GBM cannot consume a (SEQ_LEN x F_DIM) window, so each sample end e is encoded
# as the current feature vector plus its change over each lag. contig[e] already
# guarantees SEQ_LEN-1 = 191 contiguous bars behind e, so every lag (<= 180) is a
# real bar rather than one across a collector gap — asserted in the config cell.

GBM_COLS = list(features) + [f'{f_}_d{L}' for L in LAGS for f_ in features]
N_COL = len(GBM_COLS)

def make_X(ends):
    """(len(ends), F_DIM*(1+len(LAGS))) float32 design matrix for the given ends."""
    ends = np.asarray(ends)
    out = np.empty((len(ends), N_COL), dtype=np.float32)
    cur = X_raw[ends]
    out[:, :F_DIM] = cur
    for k, L in enumerate(LAGS):
        out[:, F_DIM * (k + 1):F_DIM * (k + 2)] = cur - X_raw[ends - L]
    return out

def split_ends(f_, h):
    """train / val / test ends for fold f_ at horizon h, all long-target-valid."""
    v = contig & LONG_OK[h]
    tr = ends_in(0, f_['train_hi']);  tr = tr[v[tr]]
    va = ends_in(*f_['val']);         va = va[v[va]]
    te = ends_in(*f_['test']);        te = te[v[te]]
    return tr, va, te

print(f'design matrix: {F_DIM} features x (1 + {len(LAGS)} lags) = {N_COL} columns')
for h in LONG_H:
    tr, va, te = split_ends(folds[0], h)
    print(f'  h{h:<5} fold 0:  train {len(tr):>9,}   val {len(va):>8,}   test {len(te):>8,}')


In [ ]:
# ── Cell G3: walk-forward GBM (stage 1 = mag, stage 2 = dir_all / dir_evt) ──
import xgboost as xgb

_gpu = False
try:
    import torch
    _gpu = torch.cuda.is_available()
except Exception:
    pass
DEV = 'cuda' if _gpu else 'cpu'
print(f'xgboost {xgb.__version__}   device = {DEV}\n')

def fit_gbm(Xtr, ytr, Xva, yva, seed=0):
    m = xgb.XGBRegressor(tree_method='hist', device=DEV, random_state=seed,
                         early_stopping_rounds=EARLY_ROUNDS, eval_metric='rmse',
                         **GBM_PARAMS)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return m

def cap(ix, rng):
    if TRAIN_CAP and len(ix) > TRAIN_CAP:
        return np.sort(rng.choice(ix, TRAIN_CAP, replace=False))
    return ix

# out-of-fold prediction stores, NaN outside the test blocks
P = {h: {k: np.full(n, np.nan) for k in ('mag', 'dir_all', 'dir_evt')} for h in LONG_H}
IMP = {h: {k: np.zeros(N_COL) for k in ('mag', 'dir_all', 'dir_evt')} for h in LONG_H}
BEST_IT = {h: {k: [] for k in ('mag', 'dir_all', 'dir_evt')} for h in LONG_H}

for h in LONG_H:
    y = Y_LONG[h]
    print(f'═══ h{h} ({h*WINDOW_SEC/60:.0f} min) ' + '═' * 46)
    for f_ in folds:
        rng = np.random.default_rng(1234 + f_['k'])
        tr, va, te = split_ends(f_, h)
        tr = cap(tr, rng)
        Xtr, Xva, Xte = make_X(tr), make_X(va), make_X(te)
        ytr, yva = y[tr], y[va]

        # stage 1 — magnitude (the big-move detector)
        m_mag = fit_gbm(Xtr, np.abs(ytr), Xva, np.abs(yva))
        # stage 2a — direction on everything
        m_dall = fit_gbm(Xtr, ytr, Xva, yva)
        # stage 2b — direction trained ONLY on eventful bars (train/val only)
        thr_tr = np.quantile(np.abs(ytr), 1 - EVENT_RATE)
        thr_va = np.quantile(np.abs(yva), 1 - EVENT_RATE)
        etr, eva = np.abs(ytr) >= thr_tr, np.abs(yva) >= thr_va
        m_devt = fit_gbm(Xtr[etr], ytr[etr], Xva[eva], yva[eva])

        for key, mdl in (('mag', m_mag), ('dir_all', m_dall), ('dir_evt', m_devt)):
            P[h][key][te] = mdl.predict(Xte)
            IMP[h][key] += mdl.feature_importances_
            BEST_IT[h][key].append(getattr(mdl, 'best_iteration', -1))

        print(f'  fold {f_["k"]}  train {len(tr):>8,} (evt {etr.sum():>7,})  '
              f'test {len(te):>8,}   best_iter mag/dir/evt = '
              f'{BEST_IT[h]["mag"][-1]}/{BEST_IT[h]["dir_all"][-1]}/{BEST_IT[h]["dir_evt"][-1]}')
        del Xtr, Xva, Xte; gc.collect()

for h in LONG_H:
    for k in IMP[h]:
        IMP[h][k] /= len(folds)
print('\nwalk-forward complete')


In [ ]:
# ── Cell G4: CRITERION A — is there any signal? ───────────────────────────
from scipy.stats import spearmanr

def daily_ic_t(pred_, y_, ends):
    """t-stat of the mean daily Spearman IC over test days (the project's metric)."""
    d = dt_s[ends] // 86400
    ics = []
    for u in np.unique(d):
        m = d == u
        if m.sum() < MIN_DAY_SAMPLES:
            continue
        r = spearmanr(pred_[ends][m], y_[ends][m]).statistic
        if np.isfinite(r):
            ics.append(r)
    ics = np.asarray(ics)
    if len(ics) < 5:
        return np.nan, np.nan, len(ics)
    return ics.mean() / (ics.std(ddof=1) / np.sqrt(len(ics))), ics.mean(), len(ics)

print(f'{"h":>5} {"model":>8} | {"pooled IC":>10} {"daily t":>8} {"days":>5} | '
      + '  '.join(f'{"f"+str(f_["k"]):>7}' for f_ in folds) + f' {"folds>0":>8}')
SIG = {}
for h in LONG_H:
    y = Y_LONG[h]
    for key in ('dir_all', 'dir_evt'):
        p = P[h][key]
        ok = np.isfinite(p) & np.isfinite(y)
        e = np.flatnonzero(ok)
        ic = spearmanr(p[e], y[e]).statistic
        t, icm, nday = daily_ic_t(p, y, e)
        per = []
        for f_ in folds:
            lo, hi = f_['test']
            m = e[(e >= lo) & (e < hi)]
            per.append(spearmanr(p[m], y[m]).statistic if len(m) > 100 else np.nan)
        SIG[(h, key)] = dict(ic=ic, t=t, per=per)
        print(f'{h:>5} {key:>8} | {ic:>+10.4f} {t:>+8.2f} {nday:>5} | '
              + '  '.join(f'{v:>+7.4f}' for v in per)
              + f' {sum(1 for v in per if v > 0):>6}/{len(folds)}')

print(f'\nCRITERION A: daily-IC t >= {GATE_T} and IC > 0 in >= {len(folds)-1}/{len(folds)} folds')
for (h, key), d in SIG.items():
    ok = (d['t'] >= GATE_T) and sum(1 for v in d['per'] if v > 0) >= len(folds) - 1
    print(f'  h{h:<5} {key:<8}  t={d["t"]:+6.2f}  folds>0={sum(1 for v in d["per"] if v>0)}/{len(folds)}'
          f'   -> {"PASS" if ok else "fail"}')


In [ ]:
# ── Cell G5: CRITERIA B & C — economics on the stage-1 trigger set ────────
# Stage 1 selects the top TRIG_RATE by predicted |y| within each fold; stage 2
# supplies the sign. Net bp = (2*acc - 1) * E|move| - fee, measured directly as
# mean(sign(dir) * raw_bp) - fee, with a day-clustered bootstrap CI.

def day_boot_ci(vals, days, n_boot=3000, seed=0):
    rng = np.random.default_rng(seed)
    ud = np.unique(days)
    idx = {d: np.where(days == d)[0] for d in ud}
    out = np.empty(n_boot)
    for i in range(n_boot):
        sel = np.concatenate([idx[d] for d in rng.choice(ud, len(ud), replace=True)])
        out[i] = vals[sel].mean()
    return tuple(np.percentile(out, [2.5, 97.5]))

def fold_top(score, rate, valid):
    """top-`rate` of `score` within each fold's test block (per-fold threshold)."""
    t = np.zeros(n, dtype=bool)
    for f_ in folds:
        lo, hi = f_['test']
        m = valid.copy(); m[:lo] = False; m[hi:] = False
        k = int(round(rate * m.sum()))
        if k < 1:
            continue
        ii = np.flatnonzero(m)
        s = score[ii]
        t[ii[s >= np.partition(s, -k)[-k]]] = True
    return t

ECON = {}
for h in LONG_H:
    rb, y = RAW_BP[h], Y_LONG[h]
    print(f'\n═══ h{h} ({h*WINDOW_SEC/60:.0f} min) ' + '═' * 62)
    for rate in TRIG_RATES:
        valid = np.isfinite(P[h]['mag']) & np.isfinite(rb)
        trig = fold_top(P[h]['mag'], rate, valid)
        em = np.abs(rb[trig]).mean()
        lift = em / np.abs(rb[np.isfinite(rb)]).mean()
        req = {nm: (1 + F / em) / 2 for nm, F in FEE_ROUTES}
        print(f'\n  rate {rate:.2%}   n={trig.sum():,}   E|move| {em:.2f} bp '
              f'(lift {lift:.2f}x)   required acc: '
              + '  '.join(f'{nm} {p:.3f}' for nm, p in req.items()))
        print(f'    {"stage2":>8} {"acc":>7} {"gross":>8} '
              + ' '.join(f'{"net "+nm:>9}' for nm, _ in FEE_ROUTES) + f'   {"95% CI (gross)":>22}')
        for key in ('dir_all', 'dir_evt'):
            m = trig & np.isfinite(P[h][key])
            sgn = np.sign(P[h][key][m])
            v = sgn * rb[m]
            acc = (v.mean() / np.abs(rb[m]).mean() + 1) / 2
            lo, hi = day_boot_ci(v, dt_s[m] // 86400)
            ECON[(h, rate, key)] = dict(n=int(m.sum()), em=em, acc=acc,
                                        gross=v.mean(), ci=(lo, hi), req=req)
            print(f'    {key:>8} {acc:>7.3f} {v.mean():>+8.2f} '
                  + ' '.join(f'{v.mean()-F:>+9.2f}' for _, F in FEE_ROUTES)
                  + f'   [{lo:>+8.2f}, {hi:>+8.2f}]')

print(f'\n\nCRITERION B: accuracy on top-{TRIG_RATES[0]:.1%} >= required (maker route)')
print(f'CRITERION C: net at mixed 7 bp > 0 with the day-clustered CI excluding 0')
for (h, rate, key), d in ECON.items():
    if rate != TRIG_RATES[0]:
        continue
    b = d['acc'] >= d['req']['maker']
    c = (d['gross'] - 7.0 > 0) and (d['ci'][0] - 7.0 > 0)
    print(f'  h{h:<5} {key:<8}  acc {d["acc"]:.3f} vs {d["req"]["maker"]:.3f} -> '
          f'{"PASS" if b else "fail"}   |   net7 {d["gross"]-7:+.2f} '
          f'[{d["ci"][0]-7:+.2f}, {d["ci"][1]-7:+.2f}] -> {"PASS" if c else "fail"}')


In [ ]:
# ── Cell G5b: DRIFT CONTROL — is criterion B beating skill, or beating drift? ──
# Defect found after the v1 run: criterion B compared accuracy to the FEE-breakeven
# number only. On high-volatility bars in this window the market drifts up hard, so a
# PERMANENT LONG scores 0.51-0.59 at 15 min and up to 0.68 at 1 h — i.e. B is passable
# with zero skill. run009d.offline.md section 8 solved this once already (blind
# benchmark + day-matched null); this cell ports the control the v1 probe was missing.
#
# Three references, all on the model's OWN trigger set:
#   blind      — always-long on the same bars (the drift the selection is exposed to)
#   signperm   — the model's own predicted signs SHUFFLED across the same bars, so the
#                sign multiset and therefore the drift exposure are identical. This
#                isolates "does sign ASSIGNMENT carry information", which is exactly
#                what criterion B claims.
#   volmatch   — always-long on bars chosen by CAUSAL trailing realised vol at the same
#                rate: a model-free version of the same selection.

_r1 = np.zeros(n); _ok1 = np.zeros(n, dtype=bool)
_ok1[1:] = (dt_s[1:] - dt_s[:-1]) == WINDOW_SEC
_r1[1:] = np.where(_ok1[1:], (close[1:] / close[:-1] - 1.0) * 1e4, 0.0)
TVOL = pd.Series(_r1).rolling(VOL_WINDOW, min_periods=VOL_WINDOW // 4).std().values

def sign_perm_null(sgn, rb_trig, n_draw=2000, seed=0):
    """null: same bars, same multiset of signs, shuffled."""
    rng = np.random.default_rng(seed)
    out = np.empty(n_draw)
    for i in range(n_draw):
        out[i] = (rng.permutation(sgn) * rb_trig).mean()
    return out

print(f'{"h":>5} {"rate":>7} {"stage2":>8} | {"model":>8} {"blind":>8} {"volmatch":>9} '
      f'{"excess":>8} | {"null p97.5":>10} {"pctile":>7} | {"P(up) trig":>10}')
DRIFT = {}
for h in LONG_H:
    rb = RAW_BP[h]
    for rate in TRIG_RATES:
        valid = np.isfinite(P[h]['mag']) & np.isfinite(rb)
        trig = fold_top(P[h]['mag'], rate, valid)
        blind = rb[trig].mean()
        p_up = (rb[trig] > 0).mean()
        vm = fold_top(np.where(np.isfinite(TVOL), TVOL, -np.inf), rate, valid)
        volmatch = rb[vm].mean()
        for key in ('dir_all', 'dir_evt'):
            m = trig & np.isfinite(P[h][key])
            sgn = np.sign(P[h][key][m])
            g = (sgn * rb[m]).mean()
            null = sign_perm_null(sgn, rb[m])
            pct = (null < g).mean() * 100
            DRIFT[(h, rate, key)] = dict(model=g, blind=blind, volmatch=volmatch,
                                         excess=g - blind, p_up=p_up,
                                         null_hi=np.percentile(null, 97.5), pct=pct)
            print(f'{h:>5} {rate:>7.2%} {key:>8} | {g:>+8.2f} {blind:>+8.2f} {volmatch:>+9.2f} '
                  f'{g-blind:>+8.2f} | {np.percentile(null,97.5):>+10.2f} {pct:>6.1f}% | {p_up:>10.3f}')

print()
print('REVISED CRITERION B — accuracy must beat BOTH the fee requirement AND always-long:')
for (h, rate, key), d in DRIFT.items():
    if rate != TRIG_RATES[0]:
        continue
    e = ECON[(h, rate, key)]
    need = max(e['req']['maker'], d['p_up'])
    ok = e['acc'] >= need
    print(f'  h{h:<5} {key:<8}  acc {e["acc"]:.3f}  vs  max(fee {e["req"]["maker"]:.3f}, '
          f'drift {d["p_up"]:.3f}) = {need:.3f}  -> {"PASS" if ok else "fail"}'
          f'   [excess over blind {d["excess"]:+.2f} bp, sign-perm pctile {d["pct"]:.1f}%]')
print()
print('A cell that beats the fee requirement but not always-long is measuring drift')
print('in this window, not directional skill. The sign-permutation percentile is the')
print('cleanest read: it holds the bars AND the sign multiset fixed.')


In [ ]:
# ── Cell G6: which features matter for DIRECTION (the prune, re-judged) ───
# run.009b/009d ranked features by the h18 up-head AP drop — a MAGNITUDE metric at
# a 90 s horizon — and dropped 13 as "harmful" + 5 as "dead". PRUNE_MODE='none'
# put them back. If a dropped feature ranks high here, the prune was measured on
# the wrong objective for long-horizon direction.

DROPPED = set(DROP_HARMFUL) | set(DROP_DEAD)

def base_name(col):
    for L in LAGS:
        if col.endswith(f'_d{L}'):
            return col[:-len(f'_d{L}')]
    return col

for h in LONG_H:
    print(f'\n═══ h{h} ({h*WINDOW_SEC/60:.0f} min) — dir_all gain importance ' + '═' * 28)
    imp = IMP[h]['dir_all']
    # aggregate lag variants back onto the parent feature
    agg = {}
    for c_, v in zip(GBM_COLS, imp):
        agg[base_name(c_)] = agg.get(base_name(c_), 0.0) + float(v)
    tot = sum(agg.values()) or 1.0
    rank = sorted(agg.items(), key=lambda kv: -kv[1])
    print(f'  {"#":>3} {"feature":<24} {"share":>7}  {"was pruned by run.009d?":<24}')
    for i, (f_, v) in enumerate(rank[:25], 1):
        tag = 'DROP_HARMFUL' if f_ in set(DROP_HARMFUL) else ('DROP_DEAD' if f_ in set(DROP_DEAD) else '')
        print(f'  {i:>3} {f_:<24} {v/tot*100:>6.2f}%  {tag:<24}')
    share_dropped = sum(v for f_, v in agg.items() if f_ in DROPPED) / tot
    n_top = sum(1 for f_, _ in rank[:25] if f_ in DROPPED)
    print(f'\n  the 18 run.009d-dropped features carry {share_dropped*100:.1f}% of total gain'
          f'  ({n_top}/25 of the top 25)')
    print(f'  -> {"the prune looks WRONG for this objective" if share_dropped > 18/len(features) else "the prune looks defensible here"}')


In [ ]:
# ── Cell G7: verdict + save ───────────────────────────────────────────────
print('=' * 78)
print('PROBE VERDICT')
print('=' * 78)
any_a = False
for h in LONG_H:
    for key in ('dir_all', 'dir_evt'):
        d = SIG[(h, key)]
        a = (d['t'] >= GATE_T) and sum(1 for v in d['per'] if v > 0) >= len(folds) - 1
        any_a = any_a or a
        e = ECON.get((h, TRIG_RATES[0], key))
        if e is None:
            print(f'  h{h:<5} {key:<8}  A(signal) {"PASS" if a else "fail":<4}   '
                  f'B/C not evaluated (no triggers)')
            continue
        b = e['acc'] >= e['req']['maker']
        c = (e['gross'] - 7.0 > 0) and (e['ci'][0] - 7.0 > 0)
        print(f'  h{h:<5} {key:<8}  A(signal) {"PASS" if a else "fail":<4}   '
              f'B(accuracy) {"PASS" if b else "fail":<4}   C(net) {"PASS" if c else "fail":<4}'
              f'   [IC t {d["t"]:+.2f}, acc {e["acc"]:.3f} vs {e["req"]["maker"]:.3f}]')

print()
if not any_a:
    print('  FALSIFIED: no long-horizon directional signal in the existing 76 features.')
    print('  -> Do NOT build the two-stage notebook on these inputs. The remaining')
    print('     options are new input classes (run.013 price panel) or closing the')
    print('     direction line. See horizon_economics_and_next_ideas.md section 6.7.')
else:
    print('  Criterion A passed somewhere: a long-horizon signal exists.')
    print('  Read B before celebrating — A-pass/B-fail is the run.009d-f pattern')
    print('  (signal real, too weak to trade) and is NOT a green light.')

np.savez_compressed(
    os.path.join(OUTPUT_DIR, 'gbm_probe_scores.npz'),
    **{f'p_{h}_{k}': P[h][k] for h in LONG_H for k in P[h]},
    **{f'imp_{h}_{k}': IMP[h][k] for h in LONG_H for k in IMP[h]},
    **{f'y_{h}': Y_LONG[h] for h in LONG_H},
    **{f'rawbp_{h}': RAW_BP[h] for h in LONG_H},
    dt_s=dt_s, close=close, gbm_cols=np.array(GBM_COLS),
    features=np.array(features), long_h=np.array(LONG_H),
    fold_test_lo=np.array([f_['test'][0] for f_ in folds]),
    fold_test_hi=np.array([f_['test'][1] for f_ in folds]),
)
print(f'\nsaved -> {OUTPUT_DIR}/gbm_probe_scores.npz')

import shutil
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
shutil.copy(os.path.join(OUTPUT_DIR, 'gbm_probe_scores.npz'), DRIVE_SAVE_DIR)
print(f'copied -> {DRIVE_SAVE_DIR}')
